# 1A.d Voorbeeld examenvragen — lees, voorspel, verbeter

Hieronder staat telkens een (bijna) volledige oplossing voor een examenvraag. Er zitten één of meerdere bugs in die de assert doen falen (of een fout antwoord opleveren) — vaak gaat het om een verkeerd begrip van hoe de code zich gedraagt (bv. verwarring tussen een set en een lijst, een variabele die niet gereset wordt, een voorwaarde die over de verkeerde scope gaat) in plaats van één fout symbool. Lees de code, **voorspel** de output, run, en zoek alle fouten voor je de oplossing bekijkt.

## Opgave 1: is_anagram

Deze functie moet controleren of `woord1` en `woord2` anagrammen zijn, en het aantal uitgevoerde checks teruggeven. Test met `'straat'` en `'staart'` (verwacht: `(6, True)`).

**Antwoord:** Twee samenhangende fouten. (1) `letters = set(woord1)` gebruikt een **set** in plaats van een lijst — een set houdt geen duplicaten bij, dus `'straat'` (met twee keer `'a'` en twee keer `'t'`) wordt herleid tot slechts 4 unieke letters `{'s','t','r','a'}`. Daardoor kan een letter die twee keer in `woord2` voorkomt, maar één keer "gevonden" worden — de tweede `'a'` en de tweede `'t'` in `'staart'` missen dus hun match. (2) De einduitspraak `aantal_checks == len(woord1)` is conceptueel fout: of het aantal *succesvolle* checks toevallig gelijk is aan de woordlengte zegt niets over of alle letters ook echt gematcht zijn — je moet expliciet nagaan of er geen ongematchte letters overblijven. Door bug (1) eindigt `aantal_checks` op `4` in plaats van `6`, en de assert faalt. Fix: gebruik een lijst (`list(woord1)`) zodat duplicaten meetellen, tel élke letter van `woord2` (niet enkel de gevonden) als een check, en baseer de boolean op of de lijst leeg is (`len(letters) == 0`) in plaats van op een toevallige gelijkheid.

In [1]:
def is_anagram(woord1: str, woord2: str) -> (int, bool):
    aantal_checks = 0
    if len(woord1) != len(woord2):
        return 1, False

    letters = list(woord1)
    for letter in woord2:
        aantal_checks += 1
        if letter in letters:
            letters.remove(letter)

    return aantal_checks, len(letters) == 0

woord1 = 'straat'
woord2 = 'staart'
print(is_anagram(woord1, woord2))
assert is_anagram(woord1, woord2) == (6, True)

(6, True)


## Opgave 2: vertaal_naar_binair

Deze functie moet een getal omzetten naar zijn binaire representatie als string. Test met `26` (verwacht: `'11010'`).

**Antwoord:** Twee fouten. (1) De lusvoorwaarde is `while getal > 1`, terwijl dat `getal > 0` moet zijn — met `> 1` stopt de lus zodra `getal` op `1` staat, zonder dat laatste, meest significante bit nog te verwerken. (2) Elk nieuw bit wordt achteraan toegevoegd (`binair += str(rest)`), maar het eerst berekende bit is net het *minst* significante (het restant van `getal % 2`) — dat moet dus vóóraan komen, niet achteraan. Voor `26` (`11010`) geven beide fouten samen `'0101'`: één bit te kort én in omgekeerde volgorde — makkelijk te missen omdat het resultaat er op het eerste gezicht nog als een geldige bitstring uitziet. Fix: lus laten lopen zolang `getal > 0`, en elk bit vóóraan toevoegen met `binair = str(rest) + binair`.

In [2]:
def vertaal_naar_binair(getal: int) -> str:
    binair = ''
    while getal > 0:
        rest = getal % 2
        binair = str(rest) + binair
        getal = getal // 2
    return binair

print(vertaal_naar_binair(26))
assert vertaal_naar_binair(26) == '11010'

11010


## Opgave 3: controleer_aandelen

Deze functie moet aandelen rangschikken op hun gemiddelde prijs (aflopend), als lijst van tuples `(naam, gemiddelde)` met het gemiddelde afgerond tot een integer.

**Antwoord:** `index` wordt buiten de buitenste `for`-lus op `0` gezet, in plaats van telkens opnieuw bij het verwerken van élk aandeel. Daardoor blijft `index` over de aandelen heen doortellen in plaats van bij elke nieuwe invoeging weer bij `0` te starten. Het gevolg is dat een aandeel op een volledig verkeerde positie ingevoegd kan worden: `uawhei` heeft het hoogste gemiddelde (`108`) en zou dus eerst moeten staan, maar door de opgestapelde `index` van de vorige aandelen komt het op de tweede plaats terecht. Dit is geen typfout die je aan de vergelijkingsoperator ziet — je moet begrijpen dát en waaróm `index` telkens opnieuw op `0` moet starten binnen de scope van één aandeel. Fix: verplaats `index = 0` naar binnen de `else`-tak (of naar het begin van elke iteratie van de buitenste lus), zodat elke invoeging opnieuw vanaf het begin van `uitkomst` zoekt.

In [3]:
import numpy as np

def controleer_aandelen(aandelen: dict) -> list:
    uitkomst = []
    for aandeel, prijzen in aandelen.items():
        gem = round(np.mean(prijzen))
        if len(uitkomst) == 0:
            uitkomst.append((aandeel, gem))
        else:
            index = 0
            for a, g in uitkomst:
                if gem > g:
                    uitkomst.insert(index, (aandeel, gem))
                    break
                elif index == len(uitkomst) - 1:
                    uitkomst.append((aandeel, gem))
                    break
                index += 1
    return uitkomst

aandelen = {'samsong': [10, 39], 'oppel': [50, 139], 'heiuaw': [10, 39], 'uawhei': [100, 201, 90, 39]}
oplossing = controleer_aandelen(aandelen)
print(oplossing)
assert oplossing == [('uawhei', 108), ('oppel', 94), ('samsong', 24), ('heiuaw', 24)]

[('uawhei', 108), ('oppel', 94), ('samsong', 24), ('heiuaw', 24)]


## Opgave 4: bereken_factuur

Een webshop geeft 10% korting op een product zodra een klant er **5 of meer** van bestelt. Deze functie moet het totaalbedrag van een winkelwagen berekenen (afgerond op 2 cijfers na de komma), samen met het aantal producten waarop korting werd toegepast.

**Antwoord:** Twee samenhangende fouten in de scope van de korting. (1) `aantal_besteld` telt het aantal bestelde stuks op over **alle** producten in de winkelwagen heen, terwijl de bedrijfsregel is dat de korting per product geldt (5 of meer stuks *van dat product*). Daardoor krijgt ook `hamer` (met maar 2 stuks) een korting zodra de cumulatieve teller over alle producten heen de 5 passeert — puur toeval van de volgorde waarin de producten verwerkt worden. (2) Wanneer de korting toegeslagen wordt, gebeurt dat op de **volledige lopende `totaal`** (`totaal = totaal * 0.9`) in plaats van enkel op het subtotaal van het huidige product. Omdat dit bij elk gekwalificeerd product opnieuw gebeurt, wordt eerder al toegevoegd bedrag (dat soms al gekort was) een tweede of derde keer met 10% verminderd — de korting "stapelt" zich op in plaats van netjes per productregel toegepast te worden. Het resultaat (`63.42`, met `3` "kortingen") lijkt op het eerste gezicht een plausibel totaalbedrag, wat deze fout lastig maakt om puur op het oog te herkennen. Fix: vergelijk `aantal` (niet de cumulatieve `aantal_besteld`) met de drempel, en pas de korting toe op `subtotaal` vóór je het bij `totaal` optelt.

In [4]:
def bereken_factuur(winkelwagen: dict) -> (float, int):
    """winkelwagen: {productnaam: (eenheidsprijs, aantal)}"""
    totaal = 0.0
    aantal_kortingen = 0
    for product, (prijs, aantal) in winkelwagen.items():
        subtotaal = prijs * aantal
        if aantal >= 5:
            subtotaal = subtotaal * 0.9
            aantal_kortingen += 1
        totaal += subtotaal
    return round(totaal, 2), aantal_kortingen

winkelwagen = {'schroef': (0.10, 12), 'hamer': (15.0, 2), 'verf': (8.5, 5)}
print(bereken_factuur(winkelwagen))
assert bereken_factuur(winkelwagen) == (69.33, 2)

(69.33, 2)


## Opgave 5: bereken_overuren

Een bedrijf betaalt overuren aan 1.5x het uurloon (20 euro/uur) voor elk uur boven de 38 uur per week. Deze functie berekent per werknemer de overurenvergoeding voor een gegeven week en geeft die als dict terug. Ze wordt elke week opnieuw opgeroepen met de uren van die week.

**Antwoord:** `resultaten={}` is een **mutable default argument** — die dict wordt maar één keer aangemaakt, bij het definiëren van de functie, en blijft daarna over alle latere calls heen bestaan (een bekende Python-valkuil). Bij de eerste oproep (`week1`) lijkt alles correct. Bij de tweede oproep (`week2`) wordt echter niet met een lege dict gestart: `Anke` en `Bram` staan er nog steeds in, want het is dezelfde dict-instantie als voordien. Daardoor bevat het resultaat van de assert-oproep (de derde call) ook nog `Anke` en `Bram`, en faalt de vergelijking met `{'Chris': 60.0}`. Fix: gebruik `resultaten=None` als default en maak een nieuwe dict aan binnen de functie als `resultaten is None`.

In [5]:
STANDAARDUREN = 38
UURLOON = 20

def bereken_overuren(uren_per_werknemer: dict, resultaten=None) -> dict:
    if resultaten is None:
        resultaten = {}
    for werknemer, uren in uren_per_werknemer.items():
        overuren = max(0, uren - STANDAARDUREN)
        resultaten[werknemer] = round(overuren * UURLOON * 1.5, 2)
    return resultaten

week1 = {'Anke': 42, 'Bram': 36}
week2 = {'Chris': 40}

print(bereken_overuren(week1))
print(bereken_overuren(week2))
assert bereken_overuren(week2) == {'Chris': 60.0}

{'Anke': 120.0, 'Bram': 0.0}
{'Chris': 60.0}
